# Domain D: Spatial Organization and Micro-Architecture

This notebook addresses two questions:
- **D1:** Is there spatial clustering of functionally similar neurons beyond what cell-type identity predicts?
- **D2:** Does cell-type composition vary across the columnar extent and relate to functional properties?

**Data:** Zarr multimodal stores with 3D cell coordinates (x_loc, y_loc, z_loc), cell-type labels, ΔF/F traces, and gene expression.

**Note:** Additional high-precision 3D centroids, soma volume (n_voxels), soma elongation (size_pc1/2/3_um), and orientation angle are available in the zarr multimodal stores under `morphology/mask_properties/`. See Domain H notebook for morphology-specific analyses.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, chi2_contingency
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr
from functions.tuning import compute_osi, preferred_orientation
from functions.analysis import morans_i

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# ── 3D coordinates ──
coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
print(f"Total cells: {len(obs)}")
print(f"Coordinate ranges: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], "
      f"y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}], "
      f"z=[{coords[:,2].min():.0f}, {coords[:,2].max():.0f}]")

## D2: Cell-Type Composition Across Depth and Neighborhood Effects

Analyze how the inhibitory/excitatory ratio and specific subclass distribution varies with cortical depth. Test whether local neighborhood composition (e.g., VIP density) predicts functional properties of nearby excitatory cells.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D2.1  Depth profile of cell types
# ══════════════════════════════════════════════════════════════════════

z_vals = coords[:, 2]
z_bins = np.arange(80, 360, 40)
z_centers = (z_bins[:-1] + z_bins[1:]) / 2

# Count cells per subclass per depth bin
depth_counts = {}
for sc in present_subclasses:
    sc_mask = obs['subclass_name'].values == sc
    counts, _ = np.histogram(z_vals[sc_mask], bins=z_bins)
    depth_counts[SUBCLASS_SHORT[sc]] = counts

depth_df = pd.DataFrame(depth_counts, index=z_centers)
depth_frac = depth_df.div(depth_df.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Stacked bar of proportions
ax = axes[0]
depth_frac.plot(kind='bar', stacked=True, ax=ax,
                color=[SUBCLASS_COLORS[s] for s in present_subclasses],
                width=0.8)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Fraction')
ax.set_title('D2: Cell-Type Proportions by Depth', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)

# Absolute counts
ax = axes[1]
depth_df.plot(kind='bar', ax=ax, color=[SUBCLASS_COLORS[s] for s in present_subclasses], width=0.8)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Cell Count')
ax.set_title('D2: Cell Counts by Depth', fontweight='bold')
ax.legend(fontsize=7)
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)

# E/I ratio by depth
ax = axes[2]
exc_cols = [SUBCLASS_SHORT[s] for s in present_subclasses if 'Glut' in s]
inh_cols = [SUBCLASS_SHORT[s] for s in present_subclasses if 'Gaba' in s]
exc_counts = depth_df[exc_cols].sum(axis=1)
inh_counts = depth_df[inh_cols].sum(axis=1)
ei_ratio = exc_counts / (inh_counts + 1)  # avoid div by zero
ax.bar(range(len(z_centers)), ei_ratio, color='steelblue', width=0.8)
ax.set_xticks(range(len(z_centers)))
ax.set_xticklabels([f'{int(c)}' for c in z_centers], rotation=45)
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('E/I Ratio')
ax.set_title('D2: Excitatory / Inhibitory Ratio vs Depth', fontweight='bold')

plt.tight_layout()
plt.show()

# ── 3D scatter of all cells colored by subclass ──
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')
for sc in present_subclasses:
    mask = obs['subclass_name'].values == sc
    ax.scatter(coords[mask, 0], coords[mask, 1], coords[mask, 2],
               alpha=0.4, s=10, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_zlabel('Depth (µm)')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('D2: 3D Cell Positions by Subclass', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D2.2  Neighborhood composition → functional properties
#       For each excitatory cell, count inhibitory subtypes within radius R
#       and correlate with the excitatory cell's running modulation index (RMI)
# ══════════════════════════════════════════════════════════════════════

# Compute RMI for all cells (simplified: run vs stat overall)
run_mask_trials = var['is_running'].values.astype(bool)
mean_run_all = np.nanmean(X[:, run_mask_trials], axis=1)
mean_stat_all = np.nanmean(X[:, ~run_mask_trials], axis=1)
denom_rmi = np.abs(mean_run_all) + np.abs(mean_stat_all)
denom_rmi[denom_rmi < 1e-8] = np.nan
rmi_all = (mean_run_all - mean_stat_all) / denom_rmi

# Excitatory cells
exc_subclasses = [s for s in present_subclasses if 'Glut' in s]
exc_mask = obs['subclass_name'].isin(exc_subclasses).values
exc_indices = np.where(exc_mask)[0]
exc_coords = coords[exc_mask]

# Inhibitory subclass masks
inh_types = {'Vip': '046 Vip Gaba', 'Sst': '053 Sst Gaba', 'Pvalb': '052 Pvalb Gaba', 'Lamp5': '049 Lamp5 Gaba'}

radii = [30, 50, 100]
neighborhood_results = []

for R in radii:
    for exc_i, cell_idx in enumerate(exc_indices):
        cell_coord = coords[cell_idx]
        dists = np.sqrt(np.sum((coords - cell_coord)**2, axis=1))
        neighbors = (dists < R) & (dists > 0)
        
        record = {
            'cell_idx': cell_idx,
            'radius': R,
            'rmi': rmi_all[cell_idx],
            'osi': osi[cell_idx],
            'subclass': obs['subclass_name'].iloc[cell_idx],
            'mouse_id': obs['mouse_id'].iloc[cell_idx],
        }
        for inh_name, inh_sc in inh_types.items():
            inh_in_radius = neighbors & (obs['subclass_name'].values == inh_sc)
            record[f'n_{inh_name}'] = inh_in_radius.sum()
        
        record['n_total_neighbors'] = neighbors.sum()
        neighborhood_results.append(record)

neigh_df = pd.DataFrame(neighborhood_results)

# ── Correlations: local VIP/Sst/Pvalb density → excitatory RMI ──
fig, axes = plt.subplots(len(radii), len(inh_types), figsize=(4*len(inh_types), 4*len(radii)))

for ri, R in enumerate(radii):
    sub = neigh_df[neigh_df['radius'] == R]
    for ci, (inh_name, _) in enumerate(inh_types.items()):
        ax = axes[ri, ci]
        col = f'n_{inh_name}'
        valid = ~np.isnan(sub['rmi'])
        x = sub.loc[valid, col].values
        y = sub.loc[valid, 'rmi'].values
        
        ax.scatter(x, y, alpha=0.15, s=10, c='gray')
        
        # Binned mean
        unique_counts = sorted(np.unique(x))
        for uc in unique_counts:
            mask_c = x == uc
            if mask_c.sum() >= 5:
                ax.scatter(uc, np.mean(y[mask_c]), c='red', s=60, zorder=5, edgecolors='k')
        
        if np.std(x) > 0:
            r_corr, p_corr = spearmanr(x, y)
            ax.set_title(f'{inh_name} (R={R}µm)\nρ={r_corr:.3f}, p={p_corr:.2e}',
                        fontweight='bold', fontsize=10)
        ax.set_xlabel(f'# {inh_name} within {R}µm')
        if ci == 0:
            ax.set_ylabel('Excitatory cell RMI')

plt.suptitle('D2: Local Inhibitory Density → Excitatory Running Modulation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Summary statistics ──
print("=== Neighborhood composition effects (R=50µm) ===")
sub50 = neigh_df[neigh_df['radius'] == 50]
for inh_name in inh_types:
    col = f'n_{inh_name}'
    valid = ~np.isnan(sub50['rmi'])
    r_corr, p_corr = spearmanr(sub50.loc[valid, col], sub50.loc[valid, 'rmi'])
    print(f"  {inh_name} count vs RMI: ρ={r_corr:.3f}, p={p_corr:.2e}")